In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pgmpy.models.BayesianNetwork import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pgmpy.estimators import PC
from pgmpy.estimators import MaximumLikelihoodEstimator
from imblearn.over_sampling import SMOTE
from pgmpy.estimators import HillClimbSearch, BDeuScore
from pgmpy.estimators import TreeSearch
from pgmpy.models import BayesianNetwork

from pgmpy.inference   import VariableElimination
from sklearn.metrics   import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split





In [326]:
df = pd.read_csv('Telco-Customer-Churn.csv')


In [327]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [399]:
df.isnull().sum()

customerID                               0
gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
PaperlessBilling                         0
MonthlyCharges                           0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV

In [330]:
print(df[df['TotalCharges'].str.strip() == ''])

      customerID  gender  SeniorCitizen Partner Dependents  tenure  \
488   4472-LVYGI  Female              0     Yes        Yes       0   
753   3115-CZMZD    Male              0      No        Yes       0   
936   5709-LVOEQ  Female              0     Yes        Yes       0   
1082  4367-NUYAO    Male              0     Yes        Yes       0   
1340  1371-DWPAZ  Female              0     Yes        Yes       0   
3331  7644-OMVMY    Male              0     Yes        Yes       0   
3826  3213-VVOLG    Male              0     Yes        Yes       0   
4380  2520-SGTTA  Female              0     Yes        Yes       0   
5218  2923-ARZLG    Male              0     Yes        Yes       0   
6670  4075-WKNIU  Female              0     Yes        Yes       0   
6754  2775-SEFEE    Male              0      No        Yes       0   

     PhoneService     MultipleLines InternetService       OnlineSecurity  ...  \
488            No  No phone service             DSL                  Yes  ... 

In [331]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df.fillna({'TotalCharges': df['TotalCharges'].median()}, inplace=True)


In [332]:
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [333]:
df.isnull().sum()


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [334]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [335]:
columns_to_check = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
    'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'
]

for col in columns_to_check:
    try:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            print(f"'{col}' has {zero_count} zero(s)")
    except Exception as e:
        print(f"Could not check column '{col}' — {e}")


'SeniorCitizen' has 5901 zero(s)
'tenure' has 11 zero(s)


In [336]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [337]:
le = LabelEncoder()
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn','SeniorCitizen']

for col in binary_cols:
    df[col] = df[col] == 'Yes'


In [338]:
multi_cat_cols = ['InternetService', 'MultipleLines', 'OnlineSecurity',
                  'OnlineBackup', 'DeviceProtection', 'TechSupport',
                  'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)


In [339]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 32 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   customerID                             7043 non-null   object 
 1   gender                                 7043 non-null   bool   
 2   SeniorCitizen                          7043 non-null   bool   
 3   Partner                                7043 non-null   bool   
 4   Dependents                             7043 non-null   bool   
 5   tenure                                 7043 non-null   int64  
 6   PhoneService                           7043 non-null   bool   
 7   PaperlessBilling                       7043 non-null   bool   
 8   MonthlyCharges                         7043 non-null   float64
 9   TotalCharges                           7043 non-null   float64
 10  Churn                                  7043 non-null   bool   
 11  Inte

In [341]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


In [343]:
df['Churn']

0       False
1       False
2        True
3       False
4        True
        ...  
7038    False
7039    False
7040    False
7041     True
7042    False
Name: Churn, Length: 7043, dtype: bool

In [344]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))


Churn
False    5174
True     1869
Name: count, dtype: int64
Churn
False    0.73463
True     0.26537
Name: proportion, dtype: float64


In [345]:
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn']

In [346]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [348]:
print(X.isnull().sum()[X.isnull().sum() > 0])


Series([], dtype: int64)


In [349]:
print(y_resampled.value_counts())
print(y_resampled.value_counts(normalize=True))

Churn
False    5174
True     5174
Name: count, dtype: int64
Churn
False    0.5
True     0.5
Name: proportion, dtype: float64


In [350]:
X_resampled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10348 entries, 0 to 10347
Data columns (total 30 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 10348 non-null  bool   
 1   SeniorCitizen                          10348 non-null  bool   
 2   Partner                                10348 non-null  bool   
 3   Dependents                             10348 non-null  bool   
 4   tenure                                 10348 non-null  int64  
 5   PhoneService                           10348 non-null  bool   
 6   PaperlessBilling                       10348 non-null  bool   
 7   MonthlyCharges                         10348 non-null  float64
 8   TotalCharges                           10348 non-null  float64
 9   InternetService_Fiber optic            10348 non-null  bool   
 10  InternetService_No                     10348 non-null  bool   
 11  Mu

In [351]:
X_resampled['tenure'].isnull().sum()

0

In [352]:
X_resampled['tenure'] = pd.cut(
    X_resampled['tenure'], 
    bins=[-1, 12, 24, 48, 72], 
    labels=['0-12', '13-24', '25-48', '49-72']
)

X_resampled['TotalCharges'] = pd.qcut(
    X_resampled['TotalCharges'], 
    q=3, 
    labels=['low', 'medium', 'high']
)

X_resampled['MonthlyCharges'] = pd.qcut(
    X_resampled['MonthlyCharges'], 
    q=3, 
    labels=['low', 'medium', 'high']
)

In [353]:
X_resampled.nunique()

gender                                   1
SeniorCitizen                            1
Partner                                  2
Dependents                               2
tenure                                   4
PhoneService                             2
PaperlessBilling                         2
MonthlyCharges                           3
TotalCharges                             3
InternetService_Fiber optic              2
InternetService_No                       2
MultipleLines_No phone service           2
MultipleLines_Yes                        2
OnlineSecurity_No internet service       2
OnlineSecurity_Yes                       2
OnlineBackup_No internet service         2
OnlineBackup_Yes                         2
DeviceProtection_No internet service     2
DeviceProtection_Yes                     2
TechSupport_No internet service          2
TechSupport_Yes                          2
StreamingTV_No internet service          2
StreamingTV_Yes                          2
StreamingMo

In [ ]:
df_full = pd.concat([X_resampled, y_resampled.rename("Churn")], axis=1)


train_df, test_df = train_test_split(
    df_full,
    test_size=0.2,
    random_state=42,
    stratify=df_full["Churn"]
)


In [ ]:


infer = VariableElimination(bn_model)

evidence = {
    "SeniorCitizen": True,
    "InternetService_Fiber optic": True,
    "MonthlyCharges_bin": "high",
    "Contract_One year": False,
    "Contract_Two year": False,
    "TechSupport_Yes": False
}

print(infer.query(["Churn"], evidence=evidence))


+--------------+--------------+
| Churn        |   phi(Churn) |
+==============+==============+
| Churn(False) |       0.2291 |
+--------------+--------------+
| Churn(True)  |       0.7709 |
+--------------+--------------+


In [406]:

hc = HillClimbSearch(train_df)
model = hc.estimate(scoring_method=BDeuScore(train_df), max_indegree=10)
bn_model = BayesianNetwork(model.edges())
bn_model.fit(train_df, estimator=BayesianEstimator, prior_type="BDeu")


  0%|          | 0/1000000 [00:00<?, ?it/s]

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\estimators\base.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby([variable] + parents).size().unstack(parents)
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\estimators\base.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby([variable] + parents).size().unstack(parents)
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\estimators\base.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future vers

In [ ]:

y_test   = test_df["Churn"].astype(int).values         
X_test   = test_df.drop(columns=["Churn"])
X_test = X_test.drop(columns=['gender','SeniorCitizen'])

infer     = VariableElimination(bn_model)
proba_pos = []      
for _, row in X_test.iterrows():
    evidence = row.to_dict()            
    q = infer.query(["Churn"], evidence=evidence)
    proba_pos.append(q.values[1])     

proba_pos = np.array(proba_pos)
y_pred    = (proba_pos >= 0.5).astype(int)   


acc   = accuracy_score (y_test, y_pred)
prec  = precision_score(y_test, y_pred)
rec   = recall_score   (y_test, y_pred)
auc   = roc_auc_score  (y_test, proba_pos)

results = pd.Series(
    {"Accuracy": acc, "Precision": prec, "Recall": rec, "ROC‑AUC": auc}
).round(3)

print(results)


Accuracy     0.815
Precision    0.819
Recall       0.811
ROC‑AUC      0.903
dtype: float64


In [408]:
infer = VariableElimination(bn_model)

q_fiber = infer.query(["Churn"], evidence={"InternetService_Fiber optic": True})
q_dsl   = infer.query(["Churn"], evidence={"InternetService_Fiber optic": False})

print("Churn with Fiber Optic:", q_fiber.values[1])
print("Churn without Fiber Optic:", q_dsl.values[1])


Churn with Fiber Optic: 0.6795282360803306
Churn without Fiber Optic: 0.2871187865864626


In [409]:


q_short = infer.query(["Churn"], evidence={"tenure": "0-12"})

q_long = infer.query(["Churn"], evidence={"tenure": "49-72"})

print("Churn (0-12 months):", q_short.values[1])
print("Churn (49-72 months):", q_long.values[1])


Churn (0-12 months): 0.7127086577441238
Churn (49-72 months): 0.21901588118396131


In [410]:
q_monthly = infer.query(["Churn"], evidence={
    "Contract_One year": False,
    "Contract_Two year": False
})

q_yearly = infer.query(["Churn"], evidence={
    "Contract_One year": True,
    "Contract_Two year": False
})

q_2year = infer.query(["Churn"], evidence={
    "Contract_One year": False,
    "Contract_Two year": True
})

print("Churn monthly ", q_monthly.values[1])
print("Churn yearly:", q_yearly.values[1])
print("Churn 2year:", q_2year.values[1])


Churn monthly  0.6627665221331949
Churn yearly: 0.3238924818789839
Churn 2year: 0.07419321647574306


In [ ]:


q_electronic = infer.query(["Churn"], evidence={
    "PaymentMethod_Electronic check": True
})

q_credit = infer.query(["Churn"], evidence={
    "PaymentMethod_Credit card (automatic)": True
})

print("Churn with Electronic Check:", q_electronic.values[1])
print("Churn with Credit Card (automatic):", q_credit.values[1])


Churn with Electronic Check: 0.7389806545770949
Churn with Credit Card (automatic): 0.4335165714545738
